# Hyperparameter Tuning — Optuna
**Run AFTER data pipeline is ready. Run BEFORE final V1/V2 training.**

This notebook:
1. Runs 30 Optuna trials (2hr budget) on V1 baseline
2. Writes best hyperparams back to `config.yaml`
3. Training then reruns V1 and V2 training with tuned params

In [1]:
# Imports
from src.common_utils import load_config
from src.tuning.study import run_study
from src.tuning.write_best import write_best_to_config
print('Imports OK')

Imports OK


/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
torch.cuda.empty_cache()

In [3]:
# Quick smoke test — 2 trials to verify everything works
# Change to False to run full study
SMOKE_TEST = False

cfg = load_config(variant='v1')
print(f'Trials: {cfg.tuning.n_trials}')
print(f'Trial epochs: {cfg.tuning.trial_epochs}')
print(f'Timeout: {cfg.tuning.timeout_seconds}s')
print(f'Search space: {dict(cfg.tuning.search_space)}')

Trials: 60
Trial epochs: 25
Timeout: 7200s
Search space: {'lr': [0.0001, 0.01], 'batch_size': [8, 16, 32], 'box_weight': [5.0, 10.0], 'focal_gamma': [0.5, 2.5], 'warmup_epochs': [1, 5], 'lrf': [0.001, 0.1]}


In [4]:
# Run tuning
study = run_study(
    cfg      = cfg,
    n_trials = 2 if SMOKE_TEST else None,   # None = use config value (30)
    timeout  = 300 if SMOKE_TEST else None, # None = use config value (7200s)
)
print(f'\nBest trial: #{study.best_trial.number}')
print(f'Best mAP50: {study.best_value:.4f}')
print(f'Best params: {study.best_trial.params}')

[I 2026-05-06 12:03:30,093] Using an existing study with name 'nvd-yolov9-hp-v2' instead of creating a new one.


12:03:31 | INFO     | tuning.wandb | W&B callback active → project=nvd-snow-yolov9-optuna, entity=ADL-Group-10
12:03:31 | INFO     | tuning.study | Starting study 'nvd-yolov9-hp-v2' — 60 trials, 7200s cap, sampler=tpe, pruner=hyperband
12:03:31 | INFO     | tuning.objective | [trial 1] suggested: lr=0.0005611516415334506, batch_size=8, box_weight=5.780093202212183, focal_gamma=0.8119890406724053, warmup_epochs=1, lrf=0.08675143843171859
[trainer] YOLOv9 loaded on 0
[trainer] Using existing dataset.
New https://pypi.org/project/ultralytics/8.4.46 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.45 🚀 Python-3.11.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2080 Ti, 10822MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=5.780093202212183, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cut

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



      12/25      1.84G      2.044     0.8309     0.6596         39        320: 100% ━━━━━━━━━━━━ 757/757 7.4it/s 1:42<0.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 76/76 13.1it/s 5.8s0.1s
                   all       1203       4168     0.0277     0.0881     0.0124    0.00463

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      13/25      1.84G      2.003     0.8058     0.6556         37        320: 100% ━━━━━━━━━━━━ 757/757 7.3it/s 1:43<0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 76/76 13.0it/s 5.9s.2ss
                   all       1203       4168      0.176     0.0499     0.0271     0.0071

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      14/25      1.84G       1.96     0.7759     0.6554         48        320: 100% ━━━━━━━━━━━━ 757/757 7.3it/s 1:43<0.1ss
                 Class     I

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



      10/25      5.53G      3.513     0.9586     0.6717         21        320: 100% ━━━━━━━━━━━━ 190/190 2.6it/s 1:130.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 4.7it/s 4.0s0.2s
                   all       1203       4168     0.0606     0.0365     0.0112    0.00391

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      11/25      5.52G      3.474     0.9324     0.6695         21        320: 100% ━━━━━━━━━━━━ 190/190 2.6it/s 1:140.2sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 5.1it/s 3.8s0.2s
                   all       1203       4168     0.0447     0.0669      0.016    0.00548

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      12/25      5.54G      3.384     0.8858     0.6691         34        320: 100% ━━━━━━━━━━━━ 190/190 2.6it/s 1:140.2sss
                 Class     Image

[I 2026-05-06 13:25:41,354] Trial 2 finished with value: 0.02250634553977312 and parameters: {'lr': 0.0015930522616241021, 'batch_size': 32, 'box_weight': 9.162213204002109, 'focal_gamma': 0.9246782213565523, 'warmup_epochs': 1, 'lrf': 0.01915704647548995}. Best is trial 1 with value: 0.05122193494194603.


batch_size,▁
box_weight,▁
focal_gamma,▁
lr,▁
lrf,▁
trial_number,▁
val_map50,▁
warmup_epochs,▁
batch_size,32
box_weight,9.16221
focal_gamma,0.92468


13:25:44 | INFO     | tuning.objective | [trial 3] suggested: lr=0.0004059611610484307, batch_size=8, box_weight=8.059264473611897, focal_gamma=0.7789877213040837, warmup_epochs=2, lrf=0.03726982248607548
[trainer] YOLOv9 loaded on 0
[trainer] Using existing dataset.
New https://pypi.org/project/ultralytics/8.4.46 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.45 🚀 Python-3.11.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2080 Ti, 10822MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=8.059264473611897, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/project/outputs/yolo/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [5]:
# Provide the absolute path to the config file
config_path = "/project/config.yaml"

# Preview what will be written
diff = write_best_to_config(study, config_path=config_path, dry_run=True)

print('\nChanges to config.yaml:')
for k, (old, new) in diff.items():
    print(f'  {k}: {old} → {new}')

14:35:10 | INFO     | tuning.write_best | [dry-run] would change:
14:35:10 | INFO     | tuning.write_best |    training.lr: 0.004 → 0.0005611516415334506
14:35:10 | INFO     | tuning.write_best |    training.batch_size: 16 → 8
14:35:10 | INFO     | tuning.write_best |    training.warmup_epochs: 3 → 1
14:35:10 | INFO     | tuning.write_best |    loss.box_weight: 8.925879806965067 → 5.780093202212183
14:35:10 | INFO     | tuning.write_best |    loss.focal_gamma: 0.8993475643167195 → 0.8119890406724053

Changes to config.yaml:
  training.lr: 0.004 → 0.0005611516415334506
  training.batch_size: 16 → 8
  training.warmup_epochs: 3 → 1
  loss.box_weight: 8.925879806965067 → 5.780093202212183
  loss.focal_gamma: 0.8993475643167195 → 0.8119890406724053


In [6]:
# Write best params to config.yaml
# Skip if smoke test
if not SMOKE_TEST:
    write_best_to_config(study, config_path="/project/config.yaml")
    print('config.yaml updated!')
else:
    print('Smoke test — skipping config.yaml write.')

AttributeError: 'str' object has no attribute 'with_suffix'

In [ ]:
# Plot results
import optuna
import matplotlib.pyplot as plt

fig = optuna.visualization.matplotlib.plot_optimization_history(study)
plt.savefig('outputs/results/tuning_history.png')
print('Saved: outputs/results/tuning_history.png')

fig2 = optuna.visualization.matplotlib.plot_param_importances(study)
plt.savefig('outputs/results/tuning_param_importance.png')
print('Saved: outputs/results/tuning_param_importance.png')